# Statement analytics

**Purpose:** Use `finstack_quant.statements_analytics` to stress-test, compare, explain, and score forecasts on a statement model built with `ModelBuilder`.

**Prerequisites:** Work through **notebook 09** in this track (statement modeling) so you are comfortable building periods, value nodes, compute nodes, and evaluating with `Evaluator`.


## Analytics on statement models

A financial model is a directed graph of periods and nodes. Once you can evaluate it, analytics answers adjacent questions: *What if drivers move?* *How does a new plan differ from the old one?* *What revenue clears a target EBITDA?* *Which scenarios matter?* *Where did this number come from?* *Was the forecast any good?*

The Python bindings expose these workflows as small functions that take JSON (or simple numeric vectors) and return JSON or structured Python values—ideal for notebooks, dashboards, and automated reports.


## Shared baseline: a compact P&L

We build one demo model and serialize it with `spec.to_json()`. Evaluation produces `eval_json` via `result.to_json()`. Later calls reuse these strings so each analytics API stays a pure function of model + configuration.


In [1]:
import json

from finstack_quant.statements import Evaluator, ModelBuilder

b = ModelBuilder("analytics-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)])
b.compute("ebitda", "gross_profit - opex")
spec = b.build()
model_json = spec.to_json()
result = Evaluator().evaluate(spec)
eval_json = result.to_json()

print("Model id:", json.loads(model_json)["id"])
print("EBITDA 2025Q1 (evaluated):", json.loads(eval_json)["nodes"]["ebitda"]["2025Q1"])


Model id: analytics-demo
EBITDA 2025Q1 (evaluated): 20.0


## Sensitivity and tornado views

`run_sensitivity` perturbs explicit driver nodes and records how target metrics respond. `generate_tornado_entries` narrows that JSON to a ranked list for one metric (and optional period) — useful for plotting.

**Forecast-only semantics:** `run_sensitivity` may only override forecast periods. In this demo, `actuals_until="2025Q2"`, so sensitivities should target `2025Q3` or `2025Q4`.

**Perturbation semantics:** Each entry in `perturbations` is an **absolute replacement value** for the node in that period. The `base_value` field records the original level and is used by `generate_tornado_entries` to classify perturbations as downside (below base) or upside (above base). For example, with `base_value=115` and `perturbations=[95, ..., 135]`, each value directly replaces revenue in the selected forecast period.

In [2]:
from finstack_quant.statements_analytics import generate_tornado_entries, run_sensitivity

sensitivity_config = json.dumps({
    "mode": "Diagonal",
    "parameters": [{
        "node_id": "revenue",
        "period_id": "2025Q3",
        "base_value": 115.0,
        "perturbations": [95.0, 105.0, 110.0, 115.0, 120.0, 125.0, 135.0],
    }],
    "target_metrics": ["gross_profit", "ebitda"],
})
sens_result = run_sensitivity(model_json, sensitivity_config)

parsed_sens = json.loads(sens_result)
print(f"Sensitivity: {len(parsed_sens['scenarios'])} scenarios across {parsed_sens['config']['target_metrics']}")
for sc in parsed_sens["scenarios"]:
    pv = list(sc["parameter_values"].values())[0]
    ebitda_q3 = sc["results"]["nodes"]["ebitda"]["2025Q3"]
    print(f"  revenue@Q3 = {pv:6.0f} -> EBITDA@Q3 = {ebitda_q3:8.1f}")

tornado = generate_tornado_entries(sens_result, "ebitda", "2025Q3")
print("\nTornado entries:", json.loads(tornado))

Sensitivity: 7 scenarios across ['gross_profit', 'ebitda']
  revenue@Q3 =     95 -> EBITDA@Q3 =      4.0
  revenue@Q3 =    105 -> EBITDA@Q3 =     14.0
  revenue@Q3 =    110 -> EBITDA@Q3 =     19.0
  revenue@Q3 =    115 -> EBITDA@Q3 =     24.0
  revenue@Q3 =    120 -> EBITDA@Q3 =     29.0
  revenue@Q3 =    125 -> EBITDA@Q3 =     34.0
  revenue@Q3 =    135 -> EBITDA@Q3 =     44.0

Tornado entries: [{'parameter_id': 'revenue', 'downside': -20.0, 'upside': 20.0}]


## Variance between two evaluated models

`run_variance` compares two **evaluation** JSON payloads (baseline vs comparison) for selected metrics and periods. Build a second `ModelBuilder`, evaluate it, and pass both `to_json()` outputs plus a `VarianceConfig`.


In [3]:
from finstack_quant.statements_analytics import run_variance

b2 = ModelBuilder("comparison")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("revenue", [("2025Q1", 105.0), ("2025Q2", 115.0), ("2025Q3", 120.0), ("2025Q4", 125.0)])
b2.value("cogs", [("2025Q1", 62.0), ("2025Q2", 67.0), ("2025Q3", 70.0), ("2025Q4", 74.0)])
b2.compute("gross_profit", "revenue - cogs")
b2.value("opex", [("2025Q1", 21.0), ("2025Q2", 23.0), ("2025Q3", 24.0), ("2025Q4", 25.0)])
b2.compute("ebitda", "gross_profit - opex")
comp_result = Evaluator().evaluate(b2.build())

variance_config = json.dumps({
    "baseline_label": "base",
    "comparison_label": "upside",
    "metrics": ["gross_profit", "ebitda"],
    "periods": ["2025Q1", "2025Q2"],
})
variance = json.loads(run_variance(eval_json, comp_result.to_json(), variance_config))
print(f"Variance: {variance['baseline_label']} vs {variance['comparison_label']}")
for row in variance["rows"]:
    print(f"  {row['period']} {row['metric']:15s}  base={row['baseline']:6.1f}  cmp={row['comparison']:6.1f}  "
          f"abs={row['abs_var']:+.1f}  pct={row['pct_var']:+.1%}")


Variance: base vs upside
  2025Q1 gross_profit     base=  40.0  cmp=  43.0  abs=+3.0  pct=+7.5%
  2025Q2 gross_profit     base=  45.0  cmp=  48.0  abs=+3.0  pct=+6.7%
  2025Q1 ebitda           base=  20.0  cmp=  22.0  abs=+2.0  pct=+10.0%
  2025Q2 ebitda           base=  23.0  cmp=  25.0  abs=+2.0  pct=+8.7%


## Goal seek

`goal_seek` searches for a driver value so a target node hits a desired level in a given period. With `update_model=False`, the second tuple element is `None` (no updated model is serialized)—print the solved driver and note the omitted model.


In [4]:
from finstack_quant.statements_analytics import goal_seek

gs_result = goal_seek(
    model_json,
    target_node="ebitda",
    target_period="2025Q1",
    target_value=25.0,
    driver_node="revenue",
    driver_period="2025Q1",
    update_model=False,
)
solved_revenue, updated_model_json = gs_result
print("Solved revenue (2025Q1):", solved_revenue)
print("Updated model JSON:", "<not requested (update_model=False)>" if updated_model_json is None else updated_model_json[:400])


Solved revenue (2025Q1): 104.99999999999999
Updated model JSON: <not requested (update_model=False)>


## Scenario set evaluation

`evaluate_scenario_set` runs every named scenario against the same base model. Each scenario supplies `overrides` as **node id → scalar**; those scalars replace values only in **forecast periods** (periods after `actuals_until`). Actual periods are never overridden — they retain their original values regardless of the override.

In this model, `actuals_until="2025Q2"`, so overrides only affect Q3 and Q4. Below, upside and downside shift `revenue` in the forecast window.


In [5]:
from finstack_quant.statements_analytics import evaluate_scenario_set

scenario_set = json.dumps({
    "scenarios": {
        "upside": {"overrides": {"revenue": 130.0}},
        "downside": {"overrides": {"revenue": 100.0}},
    }
})
scenario_results = json.loads(evaluate_scenario_set(model_json, scenario_set))
for name, sr in scenario_results.items():
    rev = sr["nodes"]["revenue"]
    ebitda = sr["nodes"]["ebitda"]
    print(f"--- {name} ---")
    for p in ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]:
        print(f"  {p}  revenue={rev[p]:6.1f}  ebitda={ebitda[p]:5.1f}")
print("\nNote: Q1–Q2 (actuals) retain original values; Q3–Q4 (forecast) use overrides.")


--- upside ---
  2025Q1  revenue= 100.0  ebitda= 20.0
  2025Q2  revenue= 110.0  ebitda= 23.0
  2025Q3  revenue= 130.0  ebitda= 39.0
  2025Q4  revenue= 130.0  ebitda= 34.0
--- downside ---
  2025Q1  revenue= 100.0  ebitda= 20.0
  2025Q2  revenue= 110.0  ebitda= 23.0
  2025Q3  revenue= 100.0  ebitda=  9.0
  2025Q4  revenue= 100.0  ebitda=  4.0

Note: Q1–Q2 (actuals) retain original values; Q3–Q4 (forecast) use overrides.


## Dependencies and formula explanation

`DependencyTracer.dependency_tree()` returns an ASCII tree of upstream nodes. `direct_dependencies` lists immediate inputs. `explain_formula` returns a **Python dict** (JSON-serializable) with the resolved breakdown for one node and period—use `json.dumps` for a stable string view.


In [6]:
from finstack_quant.statements_analytics import DependencyTracer, direct_dependencies, explain_formula

deps = DependencyTracer(model_json).dependency_tree("ebitda")
print("Full dependency tree:")
print(deps)

direct = direct_dependencies(model_json, "ebitda")
print("Direct deps:", direct)

explanation = explain_formula(model_json, eval_json, "ebitda", "2025Q1")
print("Formula explanation (JSON):", json.dumps(explanation, indent=2))


Full dependency tree:
ebitda (gross_profit - opex)
gross_profit (revenue - cogs)
revenue
cogs
opex

Direct deps: ['gross_profit', 'opex']
Formula explanation (JSON): {
  "node_id": "ebitda",
  "period_id": "2025Q1",
  "final_value": 20.0,
  "node_type": "Calculated",
  "formula_text": "gross_profit - opex",
  "breakdown": [
    {
      "component": "gross_profit",
      "value": 40.0,
      "operation": null
    },
    {
      "component": "opex",
      "value": 20.0,
      "operation": null
    }
  ]
}


## Backtest forecast accuracy

`backtest_forecast` expects parallel sequences of actuals and forecasts (same length) and returns MAE, MAPE, RMSE, and count. Here we align two quarters of revenue actuals vs the baseline forecast.


In [7]:
from finstack_quant.statements_analytics import backtest_forecast

actual = [102.0, 108.0]
forecast = [100.0, 110.0]
bt = backtest_forecast(actual, forecast)
print(json.dumps(dict(bt)))


{"mae": 2.0, "mape": 1.9063180827886708, "rmse": 2.0, "n": 2}


## Additional introspection APIs

Beyond `DependencyTracer` and `explain_formula`, the analytics module provides:

- **`DependencyTracer.dependency_tree_detailed`** — ASCII tree annotated with values for a specific period
- **`all_dependencies`** — flat list of all transitive upstream nodes
- **`dependents`** — reverse lookup: which nodes depend on a given node?
- **`explain_formula_text`** — human-readable multi-line explanation

In [8]:
from finstack_quant.statements_analytics import (
    DependencyTracer,
    all_dependencies,
    dependents,
    explain_formula_text,
)

tracer = DependencyTracer(model_json)
print("=== DependencyTracer.dependency_tree_detailed (ebitda, 2025Q1) ===")
print(tracer.dependency_tree_detailed(eval_json, "ebitda", "2025Q1"))

print("=== all_dependencies(ebitda) ===")
print(all_dependencies(model_json, "ebitda"))

print("\n=== dependents(revenue) — who uses revenue? ===")
print(dependents(model_json, "revenue"))

print("\n=== explain_formula_text(ebitda, 2025Q1) ===")
print(explain_formula_text(model_json, eval_json, "ebitda", "2025Q1"))

=== DependencyTracer.dependency_tree_detailed (ebitda, 2025Q1) ===
ebitda = 20.00
gross_profit = 40.00
revenue = 100.00
cogs = 60.00
opex = 20.00

=== all_dependencies(ebitda) ===
['revenue', 'cogs', 'gross_profit', 'opex']

=== dependents(revenue) — who uses revenue? ===
['gross_profit']

=== explain_formula_text(ebitda, 2025Q1) ===
ebitda [2025Q1] = 20.00
Formula: gross_profit - opex
Type: Calculated

Components:
  gross_profit = 40.00
  opex = 20.00



### Pandas export

`StatementResult` exports to pandas DataFrames in both long and wide forms, matching the notebook workflows shown in the statement modeling notebook.

In [9]:
print("Pandas wide export:")
print(result.to_pandas_wide())
print()
print("Pandas long export (first 8 rows):")
print(result.to_pandas_long().head(8))

Pandas wide export:
period_id     2025Q1  2025Q2  2025Q3  2025Q4
opex            20.0    22.0    23.0    24.0
cogs            60.0    65.0    68.0    72.0
revenue        100.0   110.0   115.0   120.0
gross_profit    40.0    45.0    47.0    48.0
ebitda          20.0    23.0    24.0    24.0

Pandas long export (first 8 rows):
  node_id  period  value value_money currency value_type
0    opex  2025Q1   20.0        None     None     scalar
1    opex  2025Q2   22.0        None     None     scalar
2    opex  2025Q3   23.0        None     None     scalar
3    opex  2025Q4   24.0        None     None     scalar
4    cogs  2025Q1   60.0        None     None     scalar
5    cogs  2025Q2   65.0        None     None     scalar
6    cogs  2025Q3   68.0        None     None     scalar
7    cogs  2025Q4   72.0        None     None     scalar


## Mini-example: full analytics pass on the demo P&L

The cell below is self-contained: it rebuilds the baseline and comparison models, then runs sensitivity, variance, scenarios, goal seek, dependency tracing, formula explanation, and a tiny revenue backtest—mirroring how you might script a quarterly analytics pack.


In [10]:
import json

from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import (
    DependencyTracer,
    backtest_forecast,
    direct_dependencies,
    evaluate_scenario_set,
    explain_formula,
    generate_tornado_entries,
    goal_seek,
    run_sensitivity,
    run_variance,
)

# Baseline
b = ModelBuilder("analytics-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)])
b.compute("ebitda", "gross_profit - opex")
spec = b.build()
model_json = spec.to_json()
base_eval = Evaluator().evaluate(spec).to_json()

# Comparison (upside plan)
b2 = ModelBuilder("comparison")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("revenue", [("2025Q1", 105.0), ("2025Q2", 115.0), ("2025Q3", 120.0), ("2025Q4", 125.0)])
b2.value("cogs", [("2025Q1", 62.0), ("2025Q2", 67.0), ("2025Q3", 70.0), ("2025Q4", 74.0)])
b2.compute("gross_profit", "revenue - cogs")
b2.value("opex", [("2025Q1", 21.0), ("2025Q2", 23.0), ("2025Q3", 24.0), ("2025Q4", 25.0)])
b2.compute("ebitda", "gross_profit - opex")
cmp_eval = Evaluator().evaluate(b2.build()).to_json()

print("=== Sensitivity (EBITDA tornado, 2025Q3) ===")
sens_cfg = json.dumps({
    "mode": "Diagonal",
    "parameters": [{
        "node_id": "revenue",
        "period_id": "2025Q3",
        "base_value": 115.0,
        "perturbations": [95.0, 105.0, 110.0, 115.0, 120.0, 125.0, 135.0],
    }],
    "target_metrics": ["gross_profit", "ebitda"],
})
sens = run_sensitivity(model_json, sens_cfg)
print("Tornado:", json.loads(generate_tornado_entries(sens, "ebitda", "2025Q3")))

print("\n=== Variance (Q1–Q2, gross_profit & ebitda) ===")
var_cfg = json.dumps({
    "baseline_label": "base",
    "comparison_label": "upside",
    "metrics": ["gross_profit", "ebitda"],
    "periods": ["2025Q1", "2025Q2"],
})
var_result = json.loads(run_variance(base_eval, cmp_eval, var_cfg))
for row in var_result["rows"]:
    print(f"  {row['period']} {row['metric']:15s}  Δ={row['abs_var']:+.1f} ({row['pct_var']:+.1%})")

print("\n=== Scenarios (revenue shocks, forecast periods only) ===")
scen = json.dumps({
    "scenarios": {
        "upside": {"overrides": {"revenue": 130.0}},
        "downside": {"overrides": {"revenue": 100.0}},
    }
})
scen_out = json.loads(evaluate_scenario_set(model_json, scen))
for name, sr in scen_out.items():
    print(f"  {name}: EBITDA Q3={sr['nodes']['ebitda']['2025Q3']}, Q4={sr['nodes']['ebitda']['2025Q4']}")

print("\n=== Goal seek EBITDA 2025Q1 -> 25 ===")
solved, _mj = goal_seek(
    model_json,
    target_node="ebitda",
    target_period="2025Q1",
    target_value=25.0,
    driver_node="revenue",
    driver_period="2025Q1",
    update_model=False,
)
print("Solved revenue:", solved)

print("\n=== Dependencies for ebitda ===")
print(DependencyTracer(model_json).dependency_tree("ebitda"))
print("Direct:", direct_dependencies(model_json, "ebitda"))
print("Explain:", json.dumps(explain_formula(model_json, base_eval, "ebitda", "2025Q1"), indent=2))

print("\n=== Revenue forecast backtest (2 quarters) ===")
bt = backtest_forecast([102.0, 108.0], [100.0, 110.0])
print(f"MAE={bt['mae']:.2f}  MAPE={bt['mape']:.2f}%  RMSE={bt['rmse']:.2f}  n={bt['n']}")

=== Sensitivity (EBITDA tornado, 2025Q3) ===
Tornado: [{'parameter_id': 'revenue', 'downside': -20.0, 'upside': 20.0}]

=== Variance (Q1–Q2, gross_profit & ebitda) ===
  2025Q1 gross_profit     Δ=+3.0 (+7.5%)
  2025Q2 gross_profit     Δ=+3.0 (+6.7%)
  2025Q1 ebitda           Δ=+2.0 (+10.0%)
  2025Q2 ebitda           Δ=+2.0 (+8.7%)

=== Scenarios (revenue shocks, forecast periods only) ===
  upside: EBITDA Q3=39.0, Q4=34.0
  downside: EBITDA Q3=9.0, Q4=4.0

=== Goal seek EBITDA 2025Q1 -> 25 ===
Solved revenue: 104.99999999999999

=== Dependencies for ebitda ===
ebitda (gross_profit - opex)
gross_profit (revenue - cogs)
revenue
cogs
opex

Direct: ['gross_profit', 'opex']
Explain: {
  "node_id": "ebitda",
  "period_id": "2025Q1",
  "final_value": 20.0,
  "node_type": "Calculated",
  "formula_text": "gross_profit - opex",
  "breakdown": [
    {
      "component": "gross_profit",
      "value": 40.0,
      "operation": null
    },
    {
      "component": "opex",
      "value": 20.0,
      

## Monte Carlo simulation

`run_monte_carlo` runs paths using the model's forecast distributions (normal/lognormal etc.) and returns percentiles + optional path data.


In [11]:
from finstack_quant.statements_analytics import run_monte_carlo
import json

mc_cfg = json.dumps({"n_paths": 1000, "seed": 42, "percentiles": [0.05, 0.5, 0.95]})
mc_json = run_monte_carlo(model_json, mc_cfg)
mc = json.loads(mc_json)
print("MC keys:", list(mc.keys())[:5])
print("ebitda p50 sample:", mc.get("ebitda", {}).get("p50"))


MC keys: ['percentile_results', 'n_paths', 'percentiles', 'forecast_periods']
ebitda p50 sample: None


## Credit assessment (dict form)

`credit_assessment` returns a structured dict (vs the text `credit_assessment_report`).


In [12]:
from finstack_quant.statements_analytics import credit_assessment

ca = credit_assessment(eval_json, "2025Q4")
print("credit_assessment keys:", list(ca.keys()) if isinstance(ca, dict) else type(ca))


credit_assessment keys: ['as_of', 'leverage_ratio', 'interest_coverage', 'free_cash_flow', 'series']


## Checks framework

The `CheckSuiteSpec` / `run_checks*` + renderers provide structured validation (three-statement, credit underwriting, custom). They are part of the surface but the primary notebook demos for checks live in dedicated model notebooks (covenant_monitoring) or are exercised via JSON configs in other tiers. See `covenant_monitoring.ipynb` and the analytics module docs for run_three_statement_checks / render_check_report_text usage patterns.


In [ ]:
import json as _json

from finstack_quant import statements as _st
from finstack_quant.statements_analytics import (
    run_checks,
    render_check_report_text,
    render_check_report_html,
)

# A model to validate
_cb = _st.ModelBuilder("checks_demo")
_cb.periods("2024Q1..Q2", None)
_cb.value("revenue", [("2024Q1", 100_000.0), ("2024Q2", 110_000.0)])
_cb.value("cogs", [("2024Q1", 40_000.0), ("2024Q2", 44_000.0)])
_cb.compute("gross_profit", "revenue - cogs")
_check_model = _cb.build().to_json()

# A check suite: parse it into a typed CheckSuiteSpec, then run it
suite_json = _json.dumps({
    "name": "P&L sanity suite",
    "builtin_checks": [],
    "formula_checks": [{
        "id": "revenue_positive",
        "name": "Revenue must be positive",
        "category": "internal_consistency",
        "severity": "error",
        "formula": "revenue > 0",
        "message_template": "Revenue not positive in {period}",
    }],
})
suite = _st.CheckSuiteSpec.from_json(suite_json)
print(f"CheckSuiteSpec '{suite.name}': {suite.formula_check_count} formula check(s)")

report_json = run_checks(_check_model, suite_json)
report = _st.CheckReport.from_json(report_json)
print(f"CheckReport: passed={report.passed} checks={report.total_checks} errors={report.total_errors} warnings={report.total_warnings}")
print("\n--- text render ---")
print(render_check_report_text(report_json))
print("html render length:", len(render_check_report_html(report_json)))

# Mapping-driven check suites: map model nodes to standard statement lines.
from finstack_quant.statements_analytics import (
    run_three_statement_checks,
    run_credit_underwriting_checks,
)

_mb = _st.ModelBuilder("ts_demo")
_mb.periods("2024Q1..Q2", None)
for node, vals in {
    "total_assets": [1000.0, 1100.0], "total_liabilities": [600.0, 650.0],
    "total_equity": [400.0, 450.0], "cash": [100.0, 130.0],
    "retained_earnings": [200.0, 250.0], "net_income": [60.0, 70.0],
    "interest_expense": [10.0, 11.0], "ebitda": [120.0, 130.0], "total_debt": [500.0, 520.0],
}.items():
    _mb.value(node, [("2024Q1", vals[0]), ("2024Q2", vals[1])])
_ts_model = _mb.build().to_json()

ts_mapping = _json.dumps({
    "assets_nodes": ["total_assets"], "liabilities_nodes": ["total_liabilities"],
    "equity_nodes": ["total_equity"], "cash_node": "cash",
    "retained_earnings_node": "retained_earnings", "net_income_node": "net_income",
    "interest_expense_node": "interest_expense",
})
ts_report = _st.CheckReport.from_json(run_three_statement_checks(_ts_model, ts_mapping))
print(f"\nrun_three_statement_checks: checks={ts_report.total_checks} passed={ts_report.passed}")

credit_mapping = _json.dumps({
    "debt_node": "total_debt", "ebitda_node": "ebitda",
    "interest_expense_node": "interest_expense", "cash_node": "cash",
})
cu_report = _st.CheckReport.from_json(run_credit_underwriting_checks(_ts_model, credit_mapping))
print(f"run_credit_underwriting_checks: checks={cu_report.total_checks} passed={cu_report.passed}")

## Takeaways

- **Sensitivity** (`run_sensitivity`, `generate_tornado_entries`) quantifies how driver shocks flow into key metrics before you commit to a plan.
- **Variance** (`run_variance`) formalizes baseline vs alternate evaluation JSON—ideal for budget vs actual or plan versions.
- **Goal seek** finds the driver level that hits a target metric in a period, preserving the rest of the model structure.
- **Scenarios** (`evaluate_scenario_set`) batch-evaluate named override sets; overrides are node-level scalars applied only to **forecast periods** (periods after `actuals_until`).
- **Introspection** (`DependencyTracer`, `all_dependencies`, `dependents`, `explain_formula`, `explain_formula_text`) supports audit trails and IC-ready narrative.
- **Backtesting** (`backtest_forecast`) scores vector forecasts with standard error metrics.
- **Export** to **pandas** with `to_pandas_wide` and `to_pandas_long`.

**Next:** Keep building richer statement models in notebook 09’s spirit, then wire these analytics into your own reporting or scenario libraries.
